Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import kagglehub
import os

In [2]:
path = kagglehub.competition_download("playground-series-s6e10")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

# 関数の定義

In [3]:
def train_lgb(train_df, test_df, feature_cols, target_col="satisfaction"):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    
    # bool型をそのまま int（1/0）に変換
    y_train = train_df[target_col].astype(int).values
    
    # ループの前にカテゴリ型への一括変換
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',

        # 'colsample_bytree': 0.8,
        # 'subsample': 0.8,
        # 'subsample_freq': 1, # 💡 subsampleを動かす時は、1イテレーションごとにサンプリングし直すこの指定をセットで入れる
        
        'n_jobs': -1,
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f"Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}")

        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f"OOF ROC-AUC Score: {score:.5f}")

    # すべての学習が終わった後、ループの外でテストデータを一括予測する
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models

# 実行

In [4]:
%%time

feature_cols = [
    'Age',
    'Flight Distance',
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Departure Delay in Minutes',
    'Arrival Delay in Minutes',
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
]


# 関数の実行
oof, preds, models = train_lgb(train, test, feature_cols)


# 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": feature_importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(30))

Fold 1 | Best Iteration:  333 | Best AUC: 0.95822
Fold 2 | Best Iteration:  457 | Best AUC: 0.95721
Fold 3 | Best Iteration:  335 | Best AUC: 0.95965
Fold 4 | Best Iteration:  315 | Best AUC: 0.95771
Fold 5 | Best Iteration:  573 | Best AUC: 0.95932
OOF ROC-AUC Score: 0.95841


,feature,importance
0,Online boarding,1.425416e+06
1,Inflight wifi service,3.590999e+05
2,Type of Travel,3.358855e+05
3,Class,1.035555e+05
4,Customer Type,9.985377e+04
5,Inflight entertainment,8.835733e+04
6,Checkin service,6.434900e+04
7,Baggage handling,5.493365e+04
8,Seat comfort,4.124116e+04
9,On-board service,4.043073e+04


CPU times: user 6min 51s, sys: 1.35 s, total: 6min 52s
Wall time: 1min 45s


# 提出

In [5]:
submission = sample_submission.copy()
submission["satisfaction"] = preds

submission.to_csv("submission.csv", index=False)
print("submission.csv を作成しました！")


# ファイルの内容・件数・欠損値のチェック
print("\n--- 先頭 5 行の確認 ---")
print(submission.head())

print("\n--- データ件数の確認 ---")
print(f"作成した件数: {len(submission)} / 元の件数: {len(sample_submission)}")

print("\n--- 欠損値（Null）の有無チェック ---")
print(f"欠損値の数: {submission['satisfaction'].isnull().sum()}")

submission.csv を作成しました！

--- 先頭 5 行の確認 ---
       id  satisfaction
0  699635      0.212362
1  699636      0.906004
2  699637      0.906881
3  699638      0.076965
4  699639      0.046907

--- データ件数の確認 ---
作成した件数: 299844 / 元の件数: 299844

--- 欠損値（Null）の有無チェック ---
欠損値の数: 0
